In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

100%|██████████| 178M/178M [00:01<00:00, 98.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2


In [3]:
import os
print(os.listdir(path))

['PS_20174392719_1491204439457_log.csv']


In [4]:
df = pd.read_csv(os.path.join(path, "PS_20174392719_1491204439457_log.csv"), )

In [5]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df.tail()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,0


In [7]:
fraud_customers = (
    df[df["isFraud"] == 1]["type"].unique())
print(f"Say {len(fraud_customers)}")
print(f"Tipler{fraud_customers}")

Say 2
Tipler['TRANSFER' 'CASH_OUT']


In [8]:
df['isFraud'].value_counts()

,count
isFraud,
0,6354407
1,8213


In [9]:
df.shape

(6362620, 11)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [11]:
fraud     = df[df["isFraud"] == 1]          #hamısını götürek
non_fraud = df[df["isFraud"] == 0].sample(n=50_000, random_state=42)

df= pd.concat([fraud, non_fraud]).reset_index(drop=True)

In [12]:
from sklearn.model_selection import train_test_split

all_customers = df["nameOrig"].unique()
train_ids, test_ids = train_test_split(
    all_customers, test_size=0.2, random_state=42)

train_df = df[df["nameOrig"].isin(train_ids)]
test_df  = df[df["nameOrig"].isin(test_ids)]

In [13]:
#Müştəri səviyyəsində aggregation
def make_agg(df):
    return df.groupby("nameOrig").agg(
        txn_count          = ("type",           "count"),

        avg_amount         = ("amount",         "mean"),
        max_amount         = ("amount",         "max"),

        cashout_count = ("type", lambda x: (x=="CASH_OUT").sum()),
        transfer_count = ("type", lambda x: (x=="TRANSFER").sum()),
        payment_count  = ("type", lambda x: (x=="PAYMENT").sum()),

        avg_balance_before = ("oldbalanceOrg", "mean"),
        balance_drain_count= ("newbalanceOrig", lambda x: (x==0).sum()),
        dest_unchanged_count=("newbalanceDest", lambda x: (x == df.loc[x.index,"oldbalanceDest"]).sum())).reset_index()

agg_train = make_agg(train_df)
agg_test  = make_agg(test_df)

In [14]:
agg_test.shape
agg_train.shape

(46570, 10)

In [15]:
for agg in [agg_train, agg_test]:
    agg["cashout_ratio"]       = agg["cashout_count"]        / agg["txn_count"]
    agg["transfer_ratio"]      = agg["transfer_count"]       / agg["txn_count"]
    agg["balance_drain_ratio"] = agg["balance_drain_count"]  / agg["txn_count"]
    agg["dest_unchanged_ratio"]= agg["dest_unchanged_count"] / agg["txn_count"]
    agg.rename(columns={"nameOrig": "customer_id"}, inplace=True)

In [16]:
aggt= df[["nameOrig","step","type","amount",
                                 "oldbalanceOrg","newbalanceOrig",
                                 "oldbalanceDest","newbalanceDest","isFraud"]].copy()

In [17]:
aggt["balance_delta_orig"] = (aggt["newbalanceOrig"] - aggt["oldbalanceOrg"]).round(2)
aggt["balance_delta_dest"] = (aggt["newbalanceDest"] - aggt["oldbalanceDest"]).round(2)

In [18]:
aggt["amount_to_balance_ratio"] = (aggt["amount"] /(aggt["oldbalanceOrg"] + 1e-3)).round(4)

aggt["is_cashout"]   = (aggt["type"]=="CASH_OUT").astype(int)
aggt["is_transfer"]  = (aggt["type"]=="TRANSFER").astype(int)

aggt["dest_unchanged"] = (aggt["newbalanceDest"] == aggt["oldbalanceDest"]).astype(int)

In [19]:
os.makedirs("data", exist_ok=True)

#Feast üçün timestamp sütunları
# step sütununu real tarixə çevir step 1 = 2024-01-01 saat 01:00
base_date = pd.Timestamp("2024-01-01", tz="UTC")

In [20]:
for agg in [agg_train, agg_test]:
 agg["event_timestamp"] = base_date +pd.to_timedelta(
    df.groupby("nameOrig")["step"].max().reindex(
        agg["customer_id"]
    ).values, unit="h")
#son əməliyyatının step-i üzrə timestamp
 agg["created"] = pd.Timestamp.now(tz="UTC")

In [21]:
aggt["event_timestamp"] = base_date + pd.to_timedelta(
    aggt["step"], unit="h")

aggt["created"] = pd.Timestamp.now(tz="UTC")

In [22]:
def add_label(agg, raw):
    labels = (
        raw.groupby('nameOrig')['isFraud']
           .max()
           .reset_index()
           .rename(columns={'nameOrig': 'customer_id', 'isFraud': 'is_fraud_customer'})
    )
    return agg.merge(labels, on='customer_id', how='left')

agg_train = add_label(agg_train, train_df)
agg_test  = add_label(agg_test,  test_df)

In [23]:
print(f"Train fraud rate: {agg_train['is_fraud_customer'].mean()*100:.1f}%")
print(f"Test  fraud rate: {agg_test['is_fraud_customer'].mean()*100:.1f}%")

Train fraud rate: 14.1%
Test  fraud rate: 14.2%


In [24]:
float_cols = ["avg_amount","max_amount","cashout_ratio","dest_unchanged_ratio",
              "transfer_ratio",
              "balance_drain_ratio",]

for col in float_cols:
    agg[col] = agg[col].astype(float)

In [25]:
int_cols = ["txn_count","cashout_count","transfer_count",
            "balance_drain_count","dest_unchanged_count"]
for col in int_cols:
    agg[col] = agg[col].astype("int64")

In [26]:
#Parquete yazaq
agg_train.to_parquet('data/agg_train.parquet', index=False)
agg_test.to_parquet('data/agg_test.parquet',   index=False)
aggt.to_parquet("data/aggt.parquet", index=False)

In [27]:
#Yoxlayaq
for fname in ["agg_test.parquet", "agg_train.parquet","aggt.parquet"]:
 size_kb = os.path.getsize(f"data/{fname}") / 1024
 df_test = pd.read_parquet(f"data/{fname}")

 print(f" {df_test.shape},{size_kb:.0f} KB")

 (11643, 17),399 KB
 (46570, 17),1520 KB
 (58213, 17),3371 KB


In [28]:
 df_test['event_timestamp'].dtype

datetime64[ns, UTC]

In [29]:
os.makedirs("feature_repo/data", exist_ok=True)

# feature_store.yaml
# Online store SQLite (development) — productionda Redis
# Offline store file (Parquet)

In [30]:
import os

repo_path = os.path.abspath("feature_repo")
db_path   = os.path.join(repo_path, "data", "registry.db")

yaml_config = f'''project: paysim_fraud_store
registry:
  registry_type: sql
  path: sqlite:///{db_path}
provider: local
online_store:
  type: sqlite
  path: {os.path.join(repo_path, "data", "online_store.db")}
offline_store:
  type: file
entity_key_serialization_version: 3
'''


In [31]:
with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(yaml_config)

print(yaml_config)

project: paysim_fraud_store
registry:
  registry_type: sql
  path: sqlite:////content/feature_repo/data/registry.db
provider: local
online_store:
  type: sqlite
  path: /content/feature_repo/data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3



In [32]:
from pathlib import Path
actual_data_dir = os.path.abspath("data")
print("Parquet faylları", actual_data_dir)

Parquet faylları /content/data


In [33]:

features_py = f'''from datetime import timedelta
from feast import Entity, FeatureView, FeatureService, Field
from feast.types import Float64, Int64
from feast.infra.offline_stores.file_source import FileSource

data_dir = "{actual_data_dir}"

customer = Entity(
    name="customer",
    join_keys=["customer_id"],)

customer_source = FileSource(
    name="customer_features_source",
    path=data_dir + "/agg_train.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",)


customer_behavior_fv = FeatureView(
    name="customer_behavior",
    entities=[customer],
    ttl=timedelta(days=365),
    schema=[
        Field(name="txn_count",      dtype=Int64),
        Field(name="avg_amount",     dtype=Float64),
        Field(name="max_amount",     dtype=Float64),
        Field(name="cashout_count",  dtype=Int64),
        Field(name="transfer_count", dtype=Int64),
        Field(name="payment_count",  dtype=Int64),
        Field(name="cashout_ratio",  dtype=Float64),
        Field(name="transfer_ratio", dtype=Float64),
    ], source=customer_source,)

customer_risk_fv = FeatureView(
    name="customer_risk",
    entities=[customer],
    ttl=timedelta(days=365),
    schema=[
        Field(name="balance_drain_ratio", dtype=Float64),
        Field(name="avg_balance_before",  dtype=Float64),
        Field(name="dest_unchanged_ratio",dtype=Float64),
   ], source=customer_source,)

fraud_detection_svc = FeatureService(
    name="fraud_detection_v1",
    features=[customer_behavior_fv,customer_risk_fv],)
'''



In [34]:
with open("feature_repo/features.py", "w") as f:
    f.write(features_py)

In [35]:
for root, _, files in os.walk("feature_repo"):
    for f in files:
        print(f"  {os.path.join(root,f)}")

  feature_repo/features.py
  feature_repo/feature_store.yaml


In [ ]:
from feast import FeatureStore
import subprocess
store = FeatureStore(repo_path='feature_repo')

subprocess.run(["feast", "apply"], cwd="feature_repo")
store.materialize_incremental(end_date=pd.Timestamp.now(tz="UTC"))

In [ ]:

print(store.project)

print("FeatureViewlar")
for fv in store.list_feature_views():
    feats = [f.name for f in fv.features]
    print(f"  [{fv.name}]  TTL={fv.ttl}  features={feats}")

print("FeatureServicelər")
for fs in store.list_feature_services():
    print(f"  [{fs.name}]  {fs.description}")

In [ ]:
training_df = store.get_historical_features(
    entity_df=agg_train[["customer_id", "event_timestamp"]],
    features=[
        "customer_behavior:txn_count",
        "customer_behavior:avg_amount",
        "customer_behavior:max_amount",
        "customer_behavior:cashout_ratio",
        "customer_behavior:transfer_ratio",
        "customer_risk:balance_drain_ratio",
        "customer_risk:dest_unchanged_ratio",
    ]
).to_df()

# 4. Yoxla
common = set(training_df["customer_id"]) & set(agg_train["customer_id"])
print(f"Uyuşan: {len(common)}")

In [ ]:
training_df.drop(columns=["event_timestamp"]).head(5)

In [ ]:
feature_cols = [
    "txn_count", "avg_amount", "max_amount",
    "cashout_ratio", "transfer_ratio",
    "balance_drain_ratio", "dest_unchanged_ratio"
]

In [ ]:
# Train datasını label ilə birləşdirek
df_ml = training_df.merge(
    agg_train[["customer_id", "is_fraud_customer"]],
    on="customer_id",
    how="inner"
).dropna(subset=["is_fraud_customer"])

X_train = agg_train[feature_cols]
y_train = agg_train['is_fraud_customer']

# Test datası agg_test-dən gəlir (ayrı aggregation)
X_test  = agg_test[feature_cols]
y_test  = agg_test['is_fraud_customer']

In [ ]:
print(f"{df_ml['is_fraud_customer'].mean()*100:.1f}%")

In [ ]:
print(f"Train: {X_train.shape} | NaN: {y_train.isna().sum()}")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Feature importances
imps = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importances:")
for feat, imp in imps.items():
    bar = "o" * int(imp * 60)
    print(f"  {feat:<25}  {imp*100:5.1f}%  {bar}")


In [ ]:
# Model qiymətləndirməsi
from sklearn.metrics import roc_auc_score, classification_report
y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = model.predict(X_test)

if y_test.nunique() > 1:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC: {auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Normal","Risk"]))
else:
    print(f"Risk score aralıq: {y_proba.min():.4f} — {y_proba.max():.4f}")


In [ ]:
import time

# Test müştəriləri (real APIda request-dən gəlir)
test_customers = agg_train["customer_id"].tolist()[:5]
test_customers

In [ ]:
t0 = time.time()
online_feat = store.get_online_features(
    features=[
        "customer_behavior:txn_count",
        "customer_behavior:avg_amount",
        "customer_behavior:max_amount",
        "customer_behavior:cashout_ratio",
        "customer_behavior:transfer_ratio",
        "customer_risk:balance_drain_ratio",
        "customer_risk:dest_unchanged_ratio",
    ],
    entity_rows=[{"customer_id": cid} for cid in test_customers],
).to_df()
latency = (time.time() - t0) * 1000

In [ ]:
print(f"Latency: {latency:.1f}ms  ({len(test_customers)} clients)")

In [ ]:
online_feat

In [ ]:
# Risk score hesablayaq
X_infer = online_feat[feature_cols].fillna(online_feat[feature_cols].median())
scores  = model.predict_proba(X_infer)[:, 1]

In [ ]:
for cid, score in zip(test_customers, scores):
    label = "yuksek — Blok" if score > 0.6 else "orta — İzlə" if score > 0.3 else "az — Keç"
    print(f"  {cid[:15]:<16}  {score:.4f}  {label}")


In [ ]:
# FeatureService ilə online sorğu
fraud_svc = store.get_feature_service("fraud_detection_v1")
online_via_svc = store.get_online_features(
    features=fraud_svc,
    entity_rows=[{"customer_id": cid} for cid in test_customers[:3]],
).to_df()

print(f"Sutunlar: {online_via_svc.columns.tolist()}")
